# getitem-back-add-at — faded example 1: Fill the scatter-add in 1-D getitem backward

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `getitem-back-add-at`. The last cell reports your progress on the `Backprop: getitem_back via add-at` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: getitem_back via add-at` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`getitem-back-add-at`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "getitem-back-add-at"
DD_SUBTOPIC = "Backprop: getitem_back via add-at"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The backward of `out = x[idx]` scatters `grad_out` into a zeros tensor using `index_add_`, which accumulates at repeated indices instead of overwriting.

## Faded exercise 1

Implement `getitem_back(grad_out, x, idx)` for `out = x[idx]` (1-D). Allocate a zeros tensor like `x`, then scatter-add `grad_out` into it along axis 0. Complete the blanked scatter-add line.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(3)
x = t.zeros(5)
idx = t.tensor([1, 1, 4])
grad_out = t.tensor([2.0, 3.0, 9.0])

def getitem_back(grad_out, x, idx):
    grad_in = t.zeros_like(x)
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    return grad_in

print(getitem_back(grad_out, x, idx).tolist())


def _test():
    gi = getitem_back(grad_out, x, idx)
    assert gi.shape == x.shape
    # independent ground truth via autograd on the same gather
    xr = t.zeros(5, requires_grad=True)
    out = xr[idx]
    (out * grad_out).sum().backward()
    assert t.allclose(gi, xr.grad), (gi, xr.grad)
    # repeated index 1 must hold the sum 2+3=5
    assert float(gi[1]) == 5.0


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(3)
x = t.zeros(5)
idx = t.tensor([1, 1, 4])
grad_out = t.tensor([2.0, 3.0, 9.0])

def getitem_back(grad_out, x, idx):
    grad_in = t.zeros_like(x)
    grad_in.index_add_(0, idx, grad_out)
    return grad_in

print(getitem_back(grad_out, x, idx).tolist())
```
</details>